In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
import torch
from datasets import Dataset



review_data = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/cleaned_reviews.csv")
review_data["review_comment"] = review_data["review_comment"].fillna("")
review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})


train_texts, test_texts, train_labels, test_labels = train_test_split(review_data["review_comment"], review_data["recommend"], test_size=0.2, random_state=2025)

train_df = pd.DataFrame({"text": train_texts, "label": train_labels})
test_df = pd.DataFrame({"text": test_texts, "label": test_labels})


train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=2025)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize datasets
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding=True, max_length=512)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)



KeyboardInterrupt: 

In [6]:
lengths = [len(example['input_ids']) for example in train_dataset]

# Find if there is any sequence with exactly 317 tokens
has_317 = any(length == 317 for length in lengths)

print(f"At least one sequence has length 317: {has_317}")

# If you want to print the index of the first sequence with 317 tokens, do this:
if has_317:
    first_317_index = next(i for i, length in enumerate(lengths) if length == 317)
    print(f"First sequence with 317 tokens is at index: {first_317_index}")

At least one sequence has length 317: False


In [30]:
# this needed to deal with memory error, I kept running out of memroy
# it may also be issue with the enviroment since I tested the same code in jupyter note book and it als crashed so likly not an IDE issue

def get_predictions(model, data, batch_size=64):
    """ Get predictions on data for a classification model m returning predictions and true labels"""
    model.eval()
    model.to('cpu')
    all_predictions = []
    
    if len(data['input_ids']) % batch_size != 0:
        num_batches = len(data['input_ids']) // batch_size + 1
    else:
        num_batches = len(data['input_ids']) // batch_size

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(data['input_ids'])) # prevent going out of bount input_ids has same size as attention mask should
        input_ids_batch = torch.tensor(data['input_ids'][start_idx:end_idx])
        attention_mask_batch = torch.tensor(data['attention_mask'][start_idx:end_idx])
        with torch.no_grad():  # Disable gradient computation for inference
            predictions = model(input_ids_batch, attention_mask=attention_mask_batch)
            
        predictions = torch.argmax(predictions.logits, dim=-1)
        all_predictions.append(predictions)
        # print(predictions)
        # print("--------")
    # print(all_predictions)
    
    
    #Concatenate all batch predictions into one tensor since we currently have a list of lists
    all_predictions = torch.cat(all_predictions, dim=0)
    
    return all_predictions , torch.tensor(data['label'])

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
import torch
from datasets import Dataset
import logging

logger = logging.getLogger()

# Set the logging level
logger.setLevel(logging.DEBUG)

review_data = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/cleaned_reviews.csv")
review_data["review_comment"] = review_data["review_comment"].fillna("")
review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})


train_texts, test_texts, train_labels, test_labels = train_test_split(review_data["review_comment"], review_data["recommend"], test_size=0.2, random_state=2025)

train_df = pd.DataFrame({"text": train_texts, "label": train_labels})
test_df = pd.DataFrame({"text": test_texts, "label": test_labels})


train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=2025)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Bert model accepts max 512
# Tokenize datasets
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding='max_length', max_length=512)


train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)


tokenizer = BertTokenizer.from_pretrained("base_1_epoch_tokenizer_bert_model_directory_try2")

# Load the model
model = BertForSequenceClassification.from_pretrained("base_1_epoch_bert_model_directory_try2")


# predictions = predictions.numpy()
# true_labels = true_labels.numpy()

# accuracy = accuracy_score(true_labels, predictions)
# print(f"Accuracy after fine-tuning: {accuracy}")


# accuracy = accuracy_score(true_labels, predictions)
# precision = precision_score(true_labels, predictions, average='macro',zero_division=0)
# recall = recall_score(true_labels, predictions, average='macro',zero_division=0)

# # Print evaluation results
# print("\nEvaluation Results on Test Data:")
# print(f"Accuracy:  {accuracy:.4f}")
# print(f"Precision: {precision:.4f}")
# print(f"Recall:    {recall:.4f}")

/opt/anaconda3/envs/cosc410/lib/python3.10/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/opt/anaconda3/envs/cosc410/lib/python3.10/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <0B7EB158-53DC-3403-8A49-22178CAB4612> /opt/anaconda3/envs/cosc410/lib/python3.10/site-packages/torchvision/image.so
  Reason: tried: '/opt/anaconda3/envs/cosc410/lib/python3.10/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/cosc410/lib/python3.10/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/cosc410/lib/python3.10/lib-dynload/../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/cosc410/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `li

Map:   0%|          | 0/53453 [00:00<?, ? examples/s]

Map:   0%|          | 0/13364 [00:00<?, ? examples/s]

Map:   0%|          | 0/16705 [00:00<?, ? examples/s]

keep track of some commands

@ pip install --force-reinstall ipykernel


// is used to create a new Jupyter kernel for the Python environment you're currently using. Let's break it down:
https://stackoverflow.com/questions/38280739/how-to-make-conda-virtual-environments-persistent-and-available-for-tools-such-a
https://ipython.readthedocs.io/en/latest/install/kernel_install.html#kernels-for-python-2-and-3
# https://stackoverflow.com/questions/28831854/how-do-i-add-python3-kernel-to-jupyter-ipython
- python -m ipykernel install --user --name=cosc410 --display-name "Python (cosc410)" 



In [ ]:
predictions, true_labels = get_predictions(model, test_dataset,64)


: 

In [ ]:
small_test_dataset = {
    'input_ids': test_dataset['input_ids'][:10],
    'attention_mask': test_dataset['attention_mask'][:10],
    'label': test_dataset['label'][:10]
}


predictions, true_labels = get_predictions(model, small_test_dataset,5)
print("Predictions:", predictions)
print("True labels:", true_labels)

Predictions: tensor([1, 1, 0, 1, 1, 1, 1, 1, 1, 1])
True labels: tensor([1, 1, 1, 0, 0, 1, 1, 1, 1, 1])


In [ ]:

# check environment
import sys
sys.executable

'/opt/anaconda3/envs/cosc410/bin/python'

In [12]:
import torch
x = torch.rand(5, 3)
x = torch.cat(x, dim=0)
print(x)

TypeError: cat() received an invalid combination of arguments - got (Tensor, dim=int), but expected one of:
 * (tuple of Tensors tensors, int dim = 0, *, Tensor out = None)
 * (tuple of Tensors tensors, name dim, *, Tensor out = None)


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import DataCollatorWithPadding
import torch
from datasets import Dataset
from trl import IterativeSFTTrainer
from tqdm import tqdm


review_data = pd.read_csv("final_5000_labeled_reviews.csv")


review_data_test = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/cleaned_reviews.csv",nrows=5000)

print(review_data_test ["review_comment"].isna().sum())
review_data_test ["review_comment"] = review_data_test ["review_comment"].fillna("") 
review_data_test ["recommend"] = review_data_test ["recommend"].map({"Recommended": 1, "Not Recommended": 0})
review_data_test ["recommend"] = review_data_test ["recommend"].fillna(-1)



review_y_test = review_data_test["recommend"]


accuracy = accuracy_score(review_y_test, review_data ["label"] )
precision = precision_score(review_y_test, review_data ["label"]  ,average='macro')
recall = recall_score(review_y_test,review_data ["label"]  ,average='macro')

print("\nEvaluation Results on Test Data:")
print(f"Accuracy:  {accuracy}")
print(f"Precision: {precision}")
print(f"Recall:    {recall}")

0

Evaluation Results on Test Data:
Accuracy:  0.9882
Precision: 0.9553498593315691
Recall:    0.9147735240499228


In [ ]:
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from datasets import Dataset
from tqdm import tqdm
from transformers import DataCollatorWithPadding
from multiprocessing import Pool, cpu_count, set_start_method
import subprocess




def get_predictions(model, data, batch_size=64):
    """ Get predictions on data for a classification model m returning predictions and true labels"""
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # model.to(device)
    device = torch.device('cpu')
    model.to(device)
 
    all_predictions = []

    
    if len(data['input_ids']) % batch_size != 0:
        num_batches = len(data['input_ids']) // batch_size + 1
    else:
        num_batches = len(data['input_ids']) // batch_size

    for i in tqdm(range(num_batches),desc= "Prediction Progress"):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(data['input_ids'])) # prevent going out of bount input_ids has same size as attention mask should
        input_ids_batch = torch.tensor(data['input_ids'][start_idx:end_idx])
        attention_mask_batch = torch.tensor(data['attention_mask'][start_idx:end_idx])
        with torch.no_grad():  # Disable gradient computation for inference
             predictions = model(input_ids_batch,attention_mask=attention_mask_batch)
             logits = predictions.logits
             predictions = torch.argmax(logits, dim=-1) # make sure it is a ser
            
        all_predictions.append(predictions)
        # print(predictions)
        # print("--------")
    # print(all_predictions)
    
    
    #Concatenate all batch predictions into one tensor since we currently have a list of lists

    all_predictions = torch.cat(all_predictions, dim=0)
    
    return all_predictions

def bootstrap_sample(dataset, seed=None):
    n = len(dataset)
    rng = random.Random(seed)  # Local random generator
    indices = [rng.randint(0,n-1) for _ in range(n)]
    return dataset.select(indices)

def train_single_boot_model(seed, model, train_data, tokenizer,num_epochs=1):
    
     

    bootstrapped_train_data = bootstrap_sample(train_data, seed)
    data_collator = DataCollatorWithPadding(tokenizer)


    training_args = TrainingArguments(
        output_dir="test_trainer",        # Directory to save logs and model checkpoints
        num_train_epochs=num_epochs,      # Number of training epochs
    )

    # Initialize the Trainer with the model, training arguments, and datasets
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=bootstrapped_train_data ,    # Training dataset     # Evaluation dataset (for validation)
        data_collator=data_collator,
    )

    # Train the model
    trainer.train()
    return model




def train_models_in_parallel(train_data,model, num_epochs=1, tokenizer=None, num_models=1):
    """Train multiple models in parallel using bootstrapped datasets."""

    num_models = min(cpu_count(),num_models)

    seeds = [2025 + i for i in range(num_models)]

    args_list = [(seed,model, train_data, tokenizer,num_epochs) for seed in seeds]

    with Pool(num_models) as pool:
        models = pool.starmap(train_single_boot_model, args_list)

    return models


def parallel_predictions(models, data, batch_size=64):
    """ Get predictions for all models in parallel."""
    predctions_list = [(model, data, batch_size) for model in models]
    with Pool(min(cpu_count(),len(models))) as pool:
        results = pool.starmap(get_predictions, predctions_list) # unpacks the tuple first
    
    # Now you have a list of predictions for each model
    combined_predictions =  combined_binary_pred(results)  # Combine predictions of shape [num_models, num_samples, num_classes]
    return combined_predictions

def combined_binary_pred(predictions):
    print(f"predictions are {predictions}")
    predictions_tensor = torch.stack(predictions)
    print(f"changed predictions are {predictions_tensor}")
    
    # Average across models (axis 0), then apply threshold of 0.5 to get final predictions
    averaged_predictions = torch.mean(predictions_tensor.float(), dim=0)

    
    # Apply threshold of 0.5 to get binary output (1 if > 0.5, else 0) 
    combined_predictions = (averaged_predictions >= 0.5).int()   # Convert boolean to int produces a bollen list then 
    
    return combined_predictions
    

def main():
    torch.cuda.empty_cache()

    # We are still in datafre
    set_start_method('spawn', force=True)
    review_data = pd.read_csv("Machine_Learning_final/opposite_data_reviews.csv",nrows = 10)
    # review_data = pd.read_csv("Machine_Learning_final/opposite_data_reviews.csv")
    print(review_data.shape)
    
    print(review_data["review_comment"].isna().sum())
    review_data["review_comment"] = review_data["review_comment"].fillna("") 
    review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})
    
    review_data["recommend"] = review_data["recommend"].fillna(-1)
    
    
    # print(torch.cuda.get_device_name(0))

    # # Run nvidia-smi and capture the output
    # try:
    #     result = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
    #     print("nvidia-smi output:\n", result.stdout)
    # except subprocess.CalledProcessError as e:
    #     print("Error running nvidia-smi:", e.stderr)
    # print(review_data[review_data["recommend"]==-1])
    
    
    # for some reason the tyepe matter https://discuss.huggingface.co/t/valueerror-target-size-torch-size-8-must-be-the-same-as-input-size-torch-size-8-8/12133/9
    train_df = pd.DataFrame({
    "text": review_data["review_comment"], 
    "label": review_data["recommend"].apply(lambda x: int(x) if x in [0, 1, -1] else -1)
    })
    
    
    
    # Now we are working with huggingface daset 
    
    train_dataset = Dataset.from_pandas(train_df)
    
    # print(sum(label == -1 for label in train_dataset["label"]))
    # print(train_dataset["label"])
    unlabeled_mask =  train_dataset["label"] == -1
    print(unlabeled_mask)
    # Load tokenizer
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    
    # Bert model accepts max 512
    # Tokenize datasets
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding='max_length', max_length=512)
    
    train_dataset = train_dataset.map(tokenize, batched=True)
    
    # print(train_dataset)
    # Load model
    # define model
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
    
    
    
    models = train_models_in_parallel(train_dataset, model, 1, tokenizer, 1)
    train_dataset = parallel_predictions(models, train_dataset, batch_size=64)
    
    
    # Save the trained model and tokenizer
    model.save_pretrained("bootstrap_bert_model")
    tokenizer.save_pretrained("bootstrap_tokenizer")
    
    # Save the final labeled dataset
    train_dataset_df = train_dataset.to_pandas()  # Convert dataset back to pandas DataFrame
    train_dataset_df = train_dataset_df[["label","text"]]
    train_dataset_df.to_csv("final_bootstrap_labeled_reviews.csv", index=False)
    


    
    #---------Evaluate-----------
    
    
    
    train_dataset_df = pd.DataFrame(train_dataset.numpy())  # Convert dataset back to pandas DataFrame
    train_dataset_df = train_dataset_df[["label","text"]]
    train_dataset_df.to_csv("final_labeled_reviews.csv", index=False)
    review_data_test = pd.read_csv("Machine_Learning_final/cleaned_reviews.csv",nrows = 10)
    # review_data_test = pd.read_csv("Machine_Learning_final/cleaned_reviews.csv")
    
    
    
    print(review_data_test ["review_comment"].isna().sum())
    review_data_test ["review_comment"] = review_data_test ["review_comment"].fillna("") 
    review_data_test ["recommend"] = review_data_test ["recommend"].map({"Recommended": 1, "Not Recommended": 0})
    review_data_test ["recommend"] = review_data_test ["recommend"].fillna(-1)
    
    
    
    review_y_test = review_data_test["recommend"]
    
    train_dataset_frame =  train_dataset.to_pandas()
    # only the unlabled data
    mask = review_data["recommend"] == -1 
    review_y_test =  review_y_test[mask]
    
    final_data_set = train_dataset_frame[mask]
    print(review_y_test)
    print(final_data_set)
    
    # print(review_X_train.shape[0])
    # This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled       data. 
    # It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.
    
    accuracy = accuracy_score(review_y_test, final_data_set["label"] )
    precision = precision_score(review_y_test, final_data_set["label"]  ,average='macro')
    recall = recall_score(review_y_test,final_data_set["label"]  ,average='macro')
    
    
    print("\nEvaluation Results on Test Data:")
    print(f"Accuracy:  {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall:    {recall}")
    
    with open('evaluation_results_selftrain_noboot_bert_models_midway.txt', 'w') as f:
        f.write(f"Accuracy after fine-tuning: {accuracy:.4f}\n")
        f.write(f"Precision: {precision:.4f}\n")
        f.write(f"Recall: {recall:.4f}\n")
        
if __name__ == '__main__':  
    main()